## Parallel evolution signatures

## 0. General

### 0.1. Load libraries

In [1]:
import pandas as pd, numpy as np, math, json
from Bio import SeqIO
from Bio.Seq import Seq

### 0.2. General variables

In [2]:
donors = [1,2,3,4]
replicates = ["A", "B", "C"]
passages = list(range(0, 110, 10))
tech_reps = [1,2 ]
populations = ["1A", "1B", "1C", "2A", "2B", "2C", "3A", "3B", "3C", "4A", "4B", "4C"]
segments = ["1-PB2", "2-PB1", "3-PA", "4-HA", "5-NP", "6-NA", "7-M", "8-NS"]
proteins = ['PB2', 'PB1', 'PB1-F2', 'PA', 'PA-X', 'HA', 'NP', 'NA', 'M1', 'M2', 'NEP', 'NS1']


hours_per_passage = 72 #hours
generation_time = 6 #hours > assumption based on Baccam, et al. 2006

In [3]:
with open('../data/CDS_H3N2.json') as json_file:
    coding_json = json.load(json_file)
segment_nuc_cds = {r.id:r.seq for r in SeqIO.parse('../data/references/stock_consensus_cds.fasta', 'fasta')}


### 0.3. Functions


Assuming transition-transversion ratio of 3;
- [bloom, 2014](https://academic.oup.com/mbe/article/31/8/1956/2925795) estimated a transition-transversion of 3
- [Pauly, et al. 2017](https://elifesciences.org/articles/26437) assumes this to be 2.7-3.6 

code in code block below adapted from [Xue and Bloom, 2020](https://academic.oup.com/ve/article/6/1/veaa010/5739536)

In [4]:
TnTv = 3
weight_Tn = 3 * TnTv / (1 + TnTv)
weight_Tv = (3 - weight_Tn) / 2

def score_codon_position(codon, position, target_AA):
  
    score = 0
    for b in ["A", "C", "T", "G"]:
        if b == codon[position]: continue
        
        #determine if mutation was a transition or a transversion and assign weight
        weight = weight_Tn if f'{codon[position]}{b}' in ["AG", "GA", "CT", "TC"] else weight_Tv
        
        
        mutated_codon = codon[:position] + b + codon[position+1:]
        aa = mutated_codon.translate()
        if aa == target_AA:
            score += weight/3.0 #division by 3 for single codon fraction

    return score

def score_codon_for_stops(codon):
    return sum(score_codon_position(codon, i, "*") for i in range(3))

def count_stop_codon_sites(codons):
    return sum(score_codon_for_stops(codon) for codon in codons)

def score_codon_for_syn(codon):
    return sum(score_codon_position(codon, i, codon.translate()) for i in range(3))

def count_syn_codon_sites(codons):
    return sum(score_codon_for_syn(codon) for codon in codons) 

In [5]:
def simulate_gene_hits(pop_hits, nsites, n_iter=10000):
    max_possible_mult = len(pop_hits)
    sim_counts = np.zeros((n_iter, max_possible_mult), dtype=np.int32)
    for i in range(n_iter):
        sim_hits = np.zeros(nsites, dtype=np.int32)
        for j in range(len(pop_hits)):
            mutated_idx = np.random.choice(nsites, pop_hits[j], replace=False)
            sim_hits[mutated_idx] += 1
        counts = np.bincount(sim_hits)
        valid_length = min(len(counts) - 1, max_possible_mult)
        sim_counts[i, :valid_length] = counts[1:valid_length + 1]
    return list(range(1, max_possible_mult + 1)), sim_counts

def empirical_pvalue(obs_value, null_dist):
    return np.mean(null_dist >= obs_value)

def weighted_parallelism_statistic(hits_dict, min_multiplicity=2):
    #Sum of multiplicities across all sites hit >= min_multiplicity times. A site hit 3 times contributes 3, a site hit 2 times contributes 2.  This preserves the signal from highly parallel sites.

    return sum(m * v for m, v in hits_dict.items() if m >= min_multiplicity)

## 1. shared SNVS across experimental populations

### 1.1. Load data

In [6]:
#load snv summary 
snv_summary = pd.read_csv("../results/snv_summary_annotated_linkage_bias.csv", keep_default_na=False, na_values=[""]).set_index(["population", "refname", "nucpos", "snv"]).sort_index()

#create overview of non-synonymous SNVs (for convienence)
non_synonymous_dict = {segment:[] for segment in segments}
for i in snv_summary[snv_summary["non_synonymous"]==True].index.droplevel(0):
    non_synonymous_dict[i[0]].append(i[-1])
    
#load trajectories 
trajectories = pd.read_csv("../results/snv_trajectories.csv").set_index(["population", "refname", "nucpos", "snv"]).sort_index()
trajectories = trajectories.rename(columns={c:int(c) for c in trajectories.columns})

### 1.2. determine shared SNVs

In [7]:
#all SNBs
shared_snv_overview = snv_summary.index.to_frame().reset_index(drop=True).drop_duplicates(ignore_index=True)
shared_snv_overview["presence"] = [f"{row['population']}" for _, row in shared_snv_overview.iterrows()]

shared_snv_overview = shared_snv_overview.pivot(index=["refname", "nucpos", "snv"], columns="population", values="presence")
shared_list = [i for i, row in shared_snv_overview.iterrows() if len(row.dropna())>1]
not_shared_list = [i for i, row in shared_snv_overview.iterrows() if len(row.dropna())==1]

shared_vs_unqiue_mat = []
for population in populations:
    df = shared_snv_overview[shared_snv_overview[population].notna()]
    counts = []
    for pop in populations:
        if pop == population:
            counts.append(len([i for i, row in df.iterrows() if len(row.dropna())== 1]))
        else:
            counts.append(len(df[df[pop].notna()]))
            
            
    shared_vs_unqiue_mat.append([population] + counts)
    
shared_vs_unqiue_mat = pd.DataFrame.from_records(shared_vs_unqiue_mat, index="population", columns=["population"]+ populations)
shared_vs_unqiue_mat.index.name = None

shared_snv_overview = shared_snv_overview.loc[shared_list]
print (len(shared_list), len(not_shared_list))

201 1331


In [8]:
#annotate with mutation type
for (refname, nucpos, snv), row in shared_snv_overview.iterrows():
    for population in populations:
        if pd.isna(row[population]):
            continue
        
        shared_snv_overview.loc[(refname, nucpos, snv), population]  = snv_summary.loc[(population, refname, nucpos, snv)]["status"].to_list()[0]

shared_drift_only = [i for i, row in shared_snv_overview.iterrows() if all(v =="drift" for v in row.dropna())]
shared_with_drift = [i for i, row in shared_snv_overview.iterrows() if any(v =="drift" for v in row.dropna()) and not all(v =="drift" for v in row.dropna())]
shared_non_drift = [i for i, row in shared_snv_overview.iterrows() if all(v !="drift" for v in row.dropna())]



In [9]:
#non-neutral SNVs only
shared_snv_overview = snv_summary[snv_summary["status"]!="drift"].index.to_frame().reset_index(drop=True).drop_duplicates(ignore_index=True)
shared_snv_overview["presence"] = [f"{row['population']}" for _, row in shared_snv_overview.iterrows()]

shared_snv_overview = shared_snv_overview.pivot(index=["refname", "nucpos", "snv"], columns="population", values="presence")
shared_list = [i for i, row in shared_snv_overview.iterrows() if len(row.dropna())>1]
not_shared_list = [i for i, row in shared_snv_overview.iterrows() if len(row.dropna())==1]

shared_vs_unqiue_mat = []
for population in populations:
    df = shared_snv_overview[shared_snv_overview[population].notna()]
    counts = []
    for pop in populations:
        if pop == population:
            counts.append(len([i for i, row in df.iterrows() if len(row.dropna())== 1]))
        else:
            counts.append(len(df[df[pop].notna()]))
            
            
    shared_vs_unqiue_mat.append([population] + counts)
    
shared_vs_unqiue_mat = pd.DataFrame.from_records(shared_vs_unqiue_mat, index="population", columns=["population"]+ populations)
shared_vs_unqiue_mat.index.name = None

shared_snv_overview = shared_snv_overview.loc[shared_list]
print (len(shared_list), len(not_shared_list))

#annotate with mutation type
for (refname, nucpos, snv), row in shared_snv_overview.iterrows():
    for population in populations:
        if pd.isna(row[population]):
            continue
        
        shared_snv_overview.loc[(refname, nucpos, snv), population]  = snv_summary.loc[(population, refname, nucpos, snv)]["status"].to_list()[0]


11 232


In [10]:
shared_snv_overview.fillna("")

population            1A     1B         1C     2A 2B     2C     3A         3B  \
refname nucpos snv                                                              
1-PB2   1971   C1971T                                                           
        2218   G2218A                                 fixed         transient   
2-PB1   1243   C1243T                fixed                   fixed              
4-HA    46     G46A       fixed  transient                                      
        1290   G1290A            transient                                      
5-NP    450    G450A                                         fixed              
6-NA    1389   C1389T                                                           
7-M     324    G324A             transient                          transient   
        653    C653T                        fixed                               
8-NS    300    A300G      fixed                                                 
        611    G611A                                  fixed                     

population                    3C         4A         4B     4C  
refname nucpos snv                                             
1-PB2   1971   C1971T                 fixed  transient         
        2218   G2218A                 fixed                    
2-PB1   1243   C1243T                                          
4-HA    46     G46A                                            
        1290   G1290A  transient                               
5-NP    450    G450A              transient                    
6-NA    1389   C1389T      fixed      fixed                    
7-M     324    G324A                                           
        653    C653T                                    fixed  
8-NS    300    A300G              transient                    
        611    G611A                                    fixed

In [11]:
shared_table = snv_summary[["non_synonymous", "protein", "aa_mut", "status", "hitchhiker", "cooccurence", "mutation bias"]]
shared_table = shared_table.reset_index().set_index(["refname", "nucpos", "snv"]).loc[shared_snv_overview.index.to_list()]
shared_table = shared_table[(shared_table["non_synonymous"]==True)&(shared_table["status"]!="drift")]

#add drifters
df = snv_summary.reset_index().set_index(["refname", "nucpos", "snv"]).sort_index()
shared_table = df[df.index.isin(shared_table.index.tolist())][shared_table.columns.tolist()]


shared_table.reset_index().set_index(["refname", "nucpos", "snv", "protein", "aa_mut", "population"]).fillna("").drop(columns="non_synonymous").to_clipboard()
shared_table.reset_index().set_index(["refname", "nucpos", "snv", "protein", "aa_mut", "population"]).fillna("")


non_synonymous     status  \
refname nucpos snv    protein aa_mut population                              
1-PB2   2218   G2218A PB2     D740N  2B                    True      drift   
                                     2C                    True      fixed   
                                     3B                    True  transient   
                                     4A                    True      fixed   
                                     4B                    True      drift   
4-HA    46     G46A   HA      A16T   1A                    True      drift   
                                     1B                    True      fixed   
                                     1C                    True  transient   
7-M     653    C653T  M1      A218V  2A                    True      fixed   
                                     4C                    True      fixed   
8-NS    611    G611A  NEP     E52K   2C                    True      fixed   
                      NS1     R204K  2C                    True      fixed   
                      NEP     E52K   4C                    True      fixed   
                      NS1     R204K  4C                    True      fixed   

                                                hitchhiker      cooccurence  \
refname nucpos snv    protein aa_mut population                               
1-PB2   2218   G2218A PB2     D740N  2B                                       
                                     2C                     high confidence   
                                     3B                                       
                                     4A                                       
                                     4B                                       
4-HA    46     G46A   HA      A16T   1A                                       
                                     1B                                       
                                     1C                                       
7-M     653    C653T  M1      A218V  2A                                       
                                     4C                                       
8-NS    611    G611A  NEP     E52K   2C                                       
                      NS1     R204K  2C                                       
                      NEP     E52K   4C                                       
                      NS1     R204K  4C                                       

                                                mutation bias  
refname nucpos snv    protein aa_mut population                
1-PB2   2218   G2218A PB2     D740N  2B                   G>A  
                                     2C                   G>A  
                                     3B                   G>A  
                                     4A                   G>A  
                                     4B                   G>A  
4-HA    46     G46A   HA      A16T   1A                   G>A  
                                     1B                   G>A  
                                     1C                   G>A  
7-M     653    C653T  M1      A218V  2A                   C>T  
                                     4C                   C>T  
8-NS    611    G611A  NEP     E52K   2C                   G>A  
                      NS1     R204K  2C                   G>A  
                      NEP     E52K   4C                   G>A  
                      NS1     R204K  4C                   G>A

**Table S6 Shared non-neutral non-synonymous SNVs**. SNVs classified as non-neutral in at least two experimental population are included in the table. Drift classifications of the included SNVS are also reported in the table. 

In [12]:
shared_table = snv_summary[["protein", "aa_mut", "status", "hitchhiker", "cooccurence", "mutation bias", "non_synonymous"]]
shared_table =  shared_table.reset_index().set_index(["refname", "nucpos", "snv"]).loc[shared_snv_overview.index.to_list()]
shared_table = shared_table[shared_table["non_synonymous"]==False]

shared_table.reset_index().set_index(["refname","snv", "protein", "aa_mut", "population"]).fillna("").drop(columns=["nucpos","non_synonymous"]).to_clipboard()
shared_table.reset_index().set_index(["refname","snv", "protein", "aa_mut", "population"]).fillna("").drop(columns=["nucpos","non_synonymous"])

status       hitchhiker  \
refname snv    protein aa_mut population                               
1-PB2   C1971T PB2     N657N  4A              fixed  high confidence   
                              4B          transient                    
2-PB1   C1243T PB1     L415L  1C              fixed  high confidence   
                              3A              fixed                    
                              4A              drift                    
4-HA    G1290A HA      E430E  1C          transient                    
                              3A              drift                    
                              3C          transient                    
5-NP    G450A  NP      R150R  3A              fixed                    
                              4A          transient                    
                              4B              drift                    
6-NA    C1389T NA      D463D  1B              drift                    
                              3C              fixed        potential   
                              4A              fixed                    
                              4B              drift                    
7-M     G324A  M1      T108T  1A              drift                    
                              1C          transient                    
                              3B          transient                    
                              4A              drift                    
                              4B              drift                    
8-NS    A300G  NS1     R100R  1B              fixed                    
                              4A          transient                    

                                         cooccurence mutation bias  
refname snv    protein aa_mut population                            
1-PB2   C1971T PB2     N657N  4A           potential           C>T  
                              4B                               C>T  
2-PB1   C1243T PB1     L415L  1C                               C>T  
                              3A                               C>T  
                              4A                               C>T  
4-HA    G1290A HA      E430E  1C                               G>A  
                              3A                               G>A  
                              3C                               G>A  
5-NP    G450A  NP      R150R  3A                               G>A  
                              4A                               G>A  
                              4B                               G>A  
6-NA    C1389T NA      D463D  1B                               C>T  
                              3C           potential           C>T  
                              4A                               C>T  
                              4B                               C>T  
7-M     G324A  M1      T108T  1A                               G>A  
                              1C                               G>A  
                              3B                               G>A  
                              4A                               G>A  
                              4B                               G>A  
8-NS    A300G  NS1     R100R  1B           potential                
                              4A

**Table S7 Shared non-neutral synonymous SNVs.** SNVs classified as non-neutral in at least two experimental population are included in the table. Drift classifications of the included SNVS are also reported in the table.

## 2. Probability of observing multiple Hits with each gene

### 2.1. Calculate the number of available sites per mutation type

In [13]:
TnTv = 3
weight_Tn = 3 * TnTv / (1 + TnTv)
weight_Tv = (3 - weight_Tn) / 2


available_sites_per_gene = {}
for segment, sdict in coding_json.items():
    available_sites_per_gene[segment] = {}
    for gene, gdict in sdict.items():
        available_sites_per_gene[segment][gene] = {}
        
        #for each gene get the coding sequence
        gene_seq = Seq("".join([b for i,b in enumerate(segment_nuc_cds[segment]) if i+1 in gdict['nuc_cds_pos']]))
        
        #split seq into codons
        gene_codons = [gene_seq[i:i+3] for i in range(0, len(gene_seq), 3)]
        
        #get number of available sites per mutation type
        stop_sites = count_stop_codon_sites(gene_codons)
        syn_sites = count_syn_codon_sites(gene_codons)
        ns_sites = len(gene_seq) - syn_sites
        
        available_sites_per_gene[segment][gene]["stop"] = stop_sites
        available_sites_per_gene[segment][gene]["syn"] = syn_sites
        available_sites_per_gene[segment][gene]["ns"] = ns_sites
        


### 2.2. observed and simulated site-level paralellism of non-neutral snvs

##### 2.2.1. non-synonymous 

In [14]:
target_genes = ["PB2", "PB1", "PA", "HA", "NP", "NA", "M1", "M2", "NS1", "NEP"]


start_df = snv_summary[(snv_summary["protein"].isin(target_genes)) &(snv_summary["non_synonymous"] == True) &(snv_summary["status"] != "drift")].reset_index()
np.random.seed(2010202)

mutatable_site_per_gene = {}
for segment, gdict in available_sites_per_gene.items():
    for protein, sdict in gdict.items():
        if protein in target_genes:
            mutatable_site_per_gene[protein] = math.ceil(sdict["ns"])

n_hits_per_gene_per_pop = { protein: start_df[start_df["protein"] == protein].groupby("population")["snv"].count().to_dict()for protein in target_genes}
multiplicity = start_df.groupby(["protein", "nucpos"])[["population"]].nunique().rename(columns={"population": "multiplicity"})
multiplicity_hit_count = (multiplicity.reset_index().groupby(["protein", "multiplicity"])[["nucpos"]].count().reset_index().pivot(index="protein", columns="multiplicity", values="nucpos").fillna(0))

l = []
for gene in target_genes:
    nsites = mutatable_site_per_gene[gene]
    pop_hits = np.array(list(n_hits_per_gene_per_pop[gene].values()))

    # simulate
    sim_hits_cols, sim_hits = simulate_gene_hits(pop_hits, nsites, n_iter=10000)

    # observed hits dict {multiplicity: n_sites}
    obs_hits = multiplicity_hit_count.loc[gene].to_dict()

    #  binary split (count of sites) 
    sim_single = sim_hits[:, 0]
    sim_multi_count = sim_hits[:, 1:].sum(axis=1)
    obs_single = sum(v for m, v in obs_hits.items() if m == 1)
    obs_multi_count = sum(v for m, v in obs_hits.items() if m >= 2)
    pvalue_count = empirical_pvalue(obs_multi_count, sim_multi_count)

    # weighted statistic > weighted by multiplicity
    multi_indices = np.where(np.array(sim_hits_cols) >= 2)[0]

    sim_multi_cols = sim_hits[:, multi_indices]
    weights = np.array(sim_hits_cols)[multi_indices]  # Strictly guarantees alignment

    sim_weighted = (sim_multi_cols * weights).sum(axis=1)
    obs_weighted = weighted_parallelism_statistic(obs_hits, min_multiplicity=2)
    pvalue_weighted = empirical_pvalue(obs_weighted, sim_weighted)

    l.append([gene, obs_single, np.median(sim_single), (np.percentile(sim_single, 2.5), np.percentile(sim_single, 97.5)),
              obs_multi_count,np.median(sim_multi_count), (np.percentile(sim_multi_count, 2.5), np.percentile(sim_multi_count, 97.5)),
              pvalue_count, obs_weighted, np.median(sim_weighted), (np.percentile(sim_weighted, 2.5), np.percentile(sim_weighted, 97.5)),pvalue_weighted,])

observed_convergence_nonneutral_ns = pd.DataFrame.from_records(l, columns=["gene", "observed single sites", "median simulated single sites", "95% CI simulated single sites",
                                                                         "observed multi sites (count)", "median simulated multi sites (count)", "95% CI simulated multi sites (count)", "p (count)",
                                                                         "observed weighted parallelism", "median simulated weighted parallelism", "95% CI simulated weighted parallelism", "p (weighted)",])

In [15]:
observed_convergence_nonneutral_ns

,gene,observed single sites,median simulated single sites,95% CI simulated single sites,observed multi sites (count),median simulated multi sites (count),95% CI simulated multi sites (count),p (count),observed weighted parallelism,median simulated weighted parallelism,95% CI simulated weighted parallelism,p (weighted)
0,PB2,12.0,15.0,"(13.0, 15.0)",1.0,0.0,"(0.0, 1.0)",0.0560,3.0,0.0,"(0.0, 2.0)",0.0010
1,PB1,10.0,10.0,"(8.0, 10.0)",0.0,0.0,"(0.0, 1.0)",1.0000,0.0,0.0,"(0.0, 2.0)",1.0000
2,PA,19.0,19.0,"(17.0, 19.0)",0.0,0.0,"(0.0, 1.0)",1.0000,0.0,0.0,"(0.0, 2.0)",1.0000
3,HA,12.0,14.0,"(12.0, 14.0)",1.0,0.0,"(0.0, 1.0)",0.0624,2.0,0.0,"(0.0, 2.0)",0.0624
4,NP,10.0,10.0,"(8.0, 10.0)",0.0,0.0,"(0.0, 1.0)",1.0000,0.0,0.0,"(0.0, 2.0)",1.0000
5,NA,17.0,17.0,"(15.0, 17.0)",0.0,0.0,"(0.0, 1.0)",1.0000,0.0,0.0,"(0.0, 2.0)",1.0000
6,M1,3.0,5.0,"(5.0, 5.0)",1.0,0.0,"(0.0, 0.0)",0.0183,2.0,0.0,"(0.0, 0.0)",0.0183
7,M2,3.0,3.0,"(3.0, 3.0)",0.0,0.0,"(0.0, 0.0)",1.0000,0.0,0.0,"(0.0, 0.0)",1.0000
8,NS1,8.0,10.0,"(8.0, 10.0)",1.0,0.0,"(0.0, 1.0)",0.0771,2.0,0.0,"(0.0, 2.0)",0.0771
9,NEP,1.0,3.0,"(3.0, 3.0)",1.0,0.0,"(0.0, 0.0)",0.0107,2.0,0.0,"(0.0, 0.0)",0.0107


**Extended Data Table 1 Observed and simulated site-level parallelism for non-neutral non-synonymous SNVs across genes.** For each gene parallelism was assessed using two statistics: a site-count statistic, which counts the number of sites mutated in two or more populations; and a multiplicity-weighted statistic, which sums the number of populations sharing a mutated site across all parallel sites. Empirical P-values were derived from 10,000 permutations.

#### 2.2.2. Synonymous

In [16]:
target_genes = ["PB2", "PB1", "PA", "HA", "NP", "NA", "M1", "M2", "NS1", "NEP"]

start_df = snv_summary[(snv_summary["protein"].isin(target_genes)) &(snv_summary["non_synonymous"] == False) &(snv_summary["status"] != "drift")].reset_index()

mutatable_site_per_gene = {}
for segment, gdict in available_sites_per_gene.items():
    for protein, sdict in gdict.items():
        if protein in target_genes:
            mutatable_site_per_gene[protein] = math.ceil(sdict["syn"])

n_hits_per_gene_per_pop = { protein: start_df[start_df["protein"] == protein].groupby("population")["snv"].count().to_dict()for protein in target_genes}
multiplicity = start_df.groupby(["protein", "nucpos"])[["population"]].nunique().rename(columns={"population": "multiplicity"})
multiplicity_hit_count = (multiplicity.reset_index().groupby(["protein", "multiplicity"])[["nucpos"]].count().reset_index().pivot(index="protein", columns="multiplicity", values="nucpos").fillna(0))

l = []
np.random.seed(2010202)
for gene in target_genes:
    nsites = mutatable_site_per_gene[gene]
    pop_hits = np.array(list(n_hits_per_gene_per_pop[gene].values()))

    # simulate
    sim_hits_cols, sim_hits = simulate_gene_hits(pop_hits, nsites, n_iter=10000)

    # observed hits dict {multiplicity: n_sites}
    obs_hits = multiplicity_hit_count.loc[gene].to_dict()

    #  binary split (count of sites) 
    sim_single = sim_hits[:, 0]
    sim_multi_count = sim_hits[:, 1:].sum(axis=1)
    obs_single = sum(v for m, v in obs_hits.items() if m == 1)
    obs_multi_count = sum(v for m, v in obs_hits.items() if m >= 2)
    pvalue_count = empirical_pvalue(obs_multi_count, sim_multi_count)

    # weighted statistic > weighted by multiplicity
    multi_indices = np.where(np.array(sim_hits_cols) >= 2)[0]

    sim_multi_cols = sim_hits[:, multi_indices]
    weights = np.array(sim_hits_cols)[multi_indices]  # Strictly guarantees alignment

    sim_weighted = (sim_multi_cols * weights).sum(axis=1)
    obs_weighted = weighted_parallelism_statistic(obs_hits, min_multiplicity=2)
    pvalue_weighted = empirical_pvalue(obs_weighted, sim_weighted)

    l.append([gene, obs_single, np.median(sim_single), (np.percentile(sim_single, 2.5), np.percentile(sim_single, 97.5)),
              obs_multi_count,np.median(sim_multi_count), (np.percentile(sim_multi_count, 2.5), np.percentile(sim_multi_count, 97.5)),
              pvalue_count, obs_weighted, np.median(sim_weighted), (np.percentile(sim_weighted, 2.5), np.percentile(sim_weighted, 97.5)),pvalue_weighted,])

observed_convergence_nonneutral_syn = pd.DataFrame.from_records(l, columns=["gene", "observed single sites", "median simulated single sites", "95% CI simulated single sites",
                                                                         "observed multi sites (count)", "median simulated multi sites (count)", "95% CI simulated multi sites (count)", "p (count)",
                                                                         "observed weighted parallelism", "median simulated weighted parallelism", "95% CI simulated weighted parallelism", "p (weighted)",])


In [17]:
observed_convergence_nonneutral_syn

,gene,observed single sites,median simulated single sites,95% CI simulated single sites,observed multi sites (count),median simulated multi sites (count),95% CI simulated multi sites (count),p (count),observed weighted parallelism,median simulated weighted parallelism,95% CI simulated weighted parallelism,p (weighted)
0,PB2,21.0,23.0,"(19.0, 23.0)",1.0,0.0,"(0.0, 2.0)",0.3014,2.0,0.0,"(0.0, 4.0)",0.3014
1,PB1,15.0,17.0,"(15.0, 17.0)",1.0,0.0,"(0.0, 1.0)",0.1854,2.0,0.0,"(0.0, 2.0)",0.1854
2,PA,26.0,28.0,"(24.0, 28.0)",1.0,0.0,"(0.0, 2.0)",0.4374,2.0,0.0,"(0.0, 4.0)",0.4374
3,HA,20.0,22.0,"(18.0, 22.0)",1.0,0.0,"(0.0, 2.0)",0.3703,2.0,0.0,"(0.0, 4.0)",0.3703
4,NP,20.0,22.0,"(18.0, 22.0)",1.0,0.0,"(0.0, 2.0)",0.4056,2.0,0.0,"(0.0, 4.0)",0.4056
5,NA,12.0,14.0,"(12.0, 14.0)",1.0,0.0,"(0.0, 1.0)",0.1845,2.0,0.0,"(0.0, 2.0)",0.1845
6,M1,10.0,12.0,"(9.0, 12.0)",1.0,0.0,"(0.0, 1.0)",0.2548,2.0,0.0,"(0.0, 3.0)",0.2548
7,M2,5.0,5.0,"(3.0, 5.0)",0.0,0.0,"(0.0, 1.0)",1.0000,0.0,0.0,"(0.0, 2.0)",1.0000
8,NS1,7.0,9.0,"(7.0, 9.0)",1.0,0.0,"(0.0, 1.0)",0.1490,2.0,0.0,"(0.0, 2.0)",0.1490
9,NEP,1.0,1.0,"(1.0, 1.0)",0.0,0.0,"(0.0, 0.0)",1.0000,0.0,0.0,"(0.0, 0.0)",1.0000


**Extended Data Table 2 Observed and simulated site-level parallelism for non-neutral synonymous SNVs across genes.** For each gene parallelism was assessed using two statistics: a site count statistic, which counts the number of sites mutated in two or more populations; and a multiplicity-weighted statistic, which sums the number of populations sharing a mutated site across all parallel sites. Empirical P-values were derived from 10,000 permutations.